# week08 · 02 — FF-neutral 토폴로지 → 힘장 라우팅 → 혼용 점검

덱 8-2 · 16장(P8-2) · 8-3 · 24장(P8-3). 토폴로지 3개(톨루엔 · 석영 셀 · 물)를 만들고 힘장별 타입을 붙인 뒤, in 파일의 혼용 위험 5가지를 ✓/✗ 로 점검한다.

실행: week08 폴더의 Claude Code 에 「`@notebooks/02_route_and_check.ipynb` 를 실행하고 결과를 요약해」(= week08 폴더에서 `python ../run_nb.py notebooks/02_route_and_check.ipynb` · 직접 보려면 JupyterLab).

In [ ]:
import subprocess, sys, json
for s in ('scripts/make_topology.py', 'scripts/route_ff.py'):
    r = subprocess.run([sys.executable, s] + (['data/topology_toluene.json','data/topology_quartz.json','data/topology_water.json'] if 'route' in s else []), capture_output=True, text=True); print(r.stdout, r.stderr[-500:])
T = json.load(open('out/routed_toluene.json'))
print('구조 해시', T['structure_hash'], '| 타입 수', T['type_counts'], '| 아티팩트 일치', T['artifact_check']['all_match'])
for r in T['routes'][:8]: print(r['id'], r['el'], 'GAFF2', r['gaff2']['type'], r['gaff2']['q'], '| COMPASS', r['compass']['type'], r['compass']['status'], '| ReaxFF', r['reaxff']['type'], r['reaxff']['q'])

같은 원자(id 1 · 방향족 C)가 GAFF2 에서는 `ca`, COMPASS 에서는 `c3a`, ReaxFF 에서는 그냥 `C` 다. 구조(좌표 · 결합)는 한 벌, 타입 열만 힘장마다 새로 채운다 — 단 COMPASS 는 9-6 이라 ClayFF(12-6) 와 한 셀에 둘 수 없어 `blocked`.

In [ ]:
for f, s in [('out/in.stack','out/stack_summary.json'), ('data/in_bad_mix.example','out/bad_summary.json')]:
    import os
    if not os.path.exists(s): s = 'out/stack_summary.json'
    r = subprocess.run([sys.executable, 'scripts/check_mix.py', f, s], capture_output=True, text=True); print('==', f); print(r.stdout)

## 확인
- `routed_toluene.json` → `artifact_check.all_match = true` (규칙으로 붙인 타입 = 연구 플랫폼 AM1-BCC 아티팩트 타입)
- `check_mix_stack.json` 5개 ✓ · `check_mix_bad_mix.json` 5개 ✗ — 어느 줄이 어느 규칙을 깨는지 `evidence` 를 읽는다
- 내 in 파일로: `python3 scripts/check_mix.py <내 in 파일> <요약 JSON>`